# Cosine stability and the arithmetic noise floor

How far can the reduction policy move a score? This is the measurement that
`docs/spec_addenda.md#g23` turns into `tau_floor`.

In [ ]:
import sys
from pathlib import Path

# Run from anywhere: notebooks/ is a sibling of src/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

In [ ]:
from tfidf_stability.analysis.noise_floor import measure_noise_floor, tau_band
from tfidf_stability.datasets.synthetic import SyntheticSpec, generate
from tfidf_stability.preprocessing.pipeline import PreprocessingPipeline
from tfidf_stability.similarity.cosine import cosine_against_corpus
from tfidf_stability.utils.numerics import Reduction
from tfidf_stability.vectorisation.tfidf import TfidfVectoriser

data = generate(SyntheticSpec(n_docs=600, vocab_size=1200, n_users=30,
                              n_exact_duplicates=12, n_twin_pairs=20))
pipeline = PreprocessingPipeline()
model = TfidfVectoriser().fit(
    [pipeline.preprocess(" ".join(d)) for d in data.documents], list(data.doc_ids)
)
documents = [model.document(i) for i in range(model.n_documents)]
queries = [
    TfidfVectoriser.transform_query(list(data.documents[i * 17 % len(data.documents)])[:6], model)
    for i in range(20)
]

floor = measure_noise_floor(model, queries)
for policy in floor.per_policy:
    print(f"{policy.policy:10} {policy.share_differing:7.2%} differ   "
          f"max {policy.max_abs:.3e}   {policy.max_ulps:.0f} ulp")
print(f"\neta       = {floor.eta:.4e}")
print(f"tau_floor = {floor.tau_floor:.4e}  (= 2 * eta)")

`NEUMAIER` is typically exactly correctly-rounded, and `PAIRWISE` is
bit-identical to `NAIVE` because its block size is 128 and no summation here is
that long. A reduction-policy sweep over short queries measures nothing.

## The band, and why it is provable

Every tau-dependent object is piecewise constant in tau, with breakpoints only
at observed gap values. So a band containing no observed gap gives *identical*
tie structure throughout -- by argument, not by sampling.

In [ ]:
exact_norms = model.matrix.row_norms(Reduction.EXACT)
vectors = [
    sorted(cosine_against_corpus(q, documents, exact_norms, Reduction.EXACT), reverse=True)
    for q in queries
]
band = tau_band(floor, vectors)

print(f"tau_floor  {band.tau_floor:.4e}")
print(f"g_min      {band.g_min:.4e}")
print(f"width      {band.decades:.2f} decades")
print(f"valid      {band.is_valid}      invariant {band.is_invariant}")
print(f"gaps inside the band: {band.n_gaps_in_band}")
print(f"exact ties {band.n_exact_ties} / positive gaps {band.n_positive_gaps}")
print(f"\nratio g_min / eta = {band.g_min / floor.eta:.3e}")
print("-> numerical error cannot close a real score gap")